# **Tratamiento de los datos**
----

### **Módulos utilizados**

In [1]:
import numpy as np
import pandas as pd
import os

from utils.utils_description import describe_df

pd.set_option('display.max_columns', 20)

## **Bloques de contenido**
1. [Introducción](#1-introducción)
2. [Objetivos de esta fase](#2-objetivos-de-esta-fase)
3. [Descripción del dataset](#3-descripción-del-dataset)
4. [Carga y primera inspección](#4-carga-y-primera-inspección)   
5. [Calidad de los datos](#5-calidad-de-los-datos)  
6. [Generación de la variable objetivo](#6-generación-de-la-variable-objetivo)  
7. [Construcción de Features RFM](#7-construcción-de-features-rfm)  
8. [Dataset final: Features + Target](#8-dataset-final-features--target)

## 1. **Introducción**
Una empresa de retail necesita **identificar** entre sus **clientes** aquellos **que tienen una alta probabilidad de volver a comprar en su establecimiento**.  
La idea es establecer unas **acciones comerciales de fidelización** para aquellos clientes con alta probabilidad de compra y **acciones de retención** para aquellos con baja probabilidad de repetir compra.

## 2. **Objetivos de esta fase**
En esta fase de **tratamiento de los datos**, se centrará en los siguientes puntos:
- Carga y primera inspección  
- Calidad de datos (nulos, duplicados, tipos)

## 3. **Descripción del dataset**
Este conjunto de datos *Online Retail II* contiene todas las transacciones realizadas por un comercio minorista online, registrado y con sede en el Reino Unido, sin tienda física, entre el *01/12/2009* y el *09/12/2011*. La empresa vende principalmente artículos de regalo únicos para cualquier ocasión. Muchos de sus clientes son mayoristas.  
Contiene tickets de compra de 24 meses recogidos en 2 hojas excel.  
|Variable|Información|Tipo|Obervaciones|
|--|--|--|--|
|Invoice|Número de factura asignado de forma única a cada transacción|Nominal|Si este código comienza con la letra "C", indica una cancelación.|
|StockCode|Código del producto (artículo) asignado de forma única a cada producto distinto.|Nominal||
|Description|Nombre del producto (artículo)|Nominal||
|Quantity|Cantidad de cada producto (artículo) por transacción|Numérico||
|InvoiceDate|Fecha y hora de la factura|Numérico|El día y la hora en que se generó la transacción|
|UnitPrice|Precio del producto por unidad en libras esterlinas (£)|Numérico||
|CustomerID|Un número entero de 5 dígitos asignado de forma única a cada cliente|Nominal||
|Country|Nombre del país donde reside el cliente|Nominal||

## 4. **Carga y primera inspección**

In [2]:
path = os.getcwd()

### 4.1. **Carga de los datos**

In [3]:
# Carga de datos
DATA_PATH = path + '\\data\\raw\\online_retail_II.xlsx'

xl = pd.ExcelFile(DATA_PATH)
df = pd.concat([pd.read_excel(DATA_PATH, sheet_name=s) for s in xl.sheet_names], ignore_index=True)

In [4]:
print(f'Filas: {len(df):,} | Columnas: {len(df.columns)}')

Filas: 1,067,371 | Columnas: 8


### 4.2. **Primera inspección**

In [5]:
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [7]:
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

In [8]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


*Primeras observaciones*:
- Aparentemente hay nulos en `Description` y `CustomerID`
- `Invoice` y `StockCode` son de tipo object ya que pueden contener letras en sus valores.
- Se observan valores negativos tanto en `Quentity` como en `Price`

## 5. **Calidad de los datos**
En esta fase se revisará y se tomará conclusiones para el tratamiento de nulos y duplicados.

In [9]:
# Descripción más exhaustiva
describe_df(df, 10, 0.2)

Clasificación sugerida para 1,067,371 filas, con un umbral para categórica de 10 sobre la cardinalidad y un umbral para númerica continua de 20.0 % sobre la cardinalidad relativa.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
Columnas,,,,,,,,
Tipo_Dato,object,object,object,int64,datetime64[us],float64,float64,str
Nulos,0,0,4382,0,0,0,243007,0
Nulos_%,0.0,0.0,0.4,0.0,0.0,0.0,22.8,0.0
Cardinalidad,53628,5305,5698,1057,47635,2807,5942,43
Cardinalidad_%,5.02,0.5,0.53,0.1,4.46,0.26,0.56,0.0
Clasificacion_sugerida,Categorica,Categorica,Categorica,Numerica_Discreta,Numerica_Discreta,Numerica_Discreta,Numerica_Discreta,Categorica


*Observaciones*
- El **porcentaje de nulos** en `Description` no representa ni el 1% de los datos, sin embargo, en `Customer ID`, sí que puede ser algo problemático.

Para el tratamiento de los valores nulos, se analizará, en primer lugar, la existencia de duplicados ya que algunos nulos puedn formar parte de esos duplicados

### 5.1. **Renombramiento de los nombres de las variales**

In [4]:
df.columns = [col.lower().replace(' ','_') for col in df.columns]

### 5.2. **Valores duplicados**

In [11]:
print(f'En el dataset hay {df.duplicated().sum():,} filas duplicadas')

En el dataset hay 34,335 filas duplicadas


In [12]:
# Se guarda en una variable para analizarlas
df_duplicated = df.loc[df.duplicated()]

In [13]:
df_duplicated.info()

<class 'pandas.DataFrame'>
Index: 34335 entries, 371 to 1067162
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   invoice      34335 non-null  object        
 1   stockcode    34335 non-null  object        
 2   description  34228 non-null  object        
 3   quantity     34335 non-null  int64         
 4   invoicedate  34335 non-null  datetime64[us]
 5   price        34335 non-null  float64       
 6   customer_id  26479 non-null  float64       
 7   country      34335 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 2.4+ MB


In [14]:
for col in ['description', 'customer_id']:
    print(f'Porcentaje de nulos de "{col}" -> {round((df_duplicated[col].isna().sum()/len(df)) * 100, 2)} %')

Porcentaje de nulos de "description" -> 0.01 %
Porcentaje de nulos de "customer_id" -> 0.74 %


*Observaciones*:
- En los valores duplicados hay valores nulos aunque no representan ni un tercio de los generales, sin embargo, se eliminarán algunos.

In [5]:
# Eliminación de los valores duplicados quedando la primera ocurrencia
df_dd = df.drop_duplicates()

### 5.3. **Eliminación de nulos en `Customer ID`**
Se eliminan los valores nulos de esta variable ya que no se puede asociar a ningún cliente y cualquier imputación que se haga puede producir sesgos.

In [6]:
df_dd = df_dd.dropna(subset=['customer_id'])
df_dd['customer_id'] = df_dd['customer_id'].astype(int) # Se convierte en int ya que estaba en float

### 5.4. **Análisis de los valores atípicos en `Quantity` y `Price`**
En primer lugar, se analizará estos valores atípicos

In [17]:
# Se guarda en otro DF
df_qp = df_dd.loc[(df_dd['quantity'] <= 0) | (df_dd['price'] <= 0)]

In [18]:
# En 'quantity', ¿hay facturas realizadas que tengan este valor?
df_qp[~(df_qp['invoice'].astype(str).str.startswith('C')) & (df_qp['quantity'] <= 0)]

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country


*Observaciones*: Se puede concluir que todos los valores negativos o 0s de `quantity` son cancelaciones, por lo que lo dejamos como está.

In [19]:
# Se filtra por 'price'
df_qp.loc[df_qp['price'] < 0] # No hay valores negativos

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country


In [20]:
df_qp[(df_qp['invoice'].astype(str).str.startswith('C')) & (df_qp['price'] == 0)] # No hay cancelaciones con precios a 0

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country


In [21]:
df_pz = df_qp.loc[df_qp['price'] == 0] # Se crea DF con el filtrado
print(f'Total de registros con "price" a 0: {len(df_pz)}')
print(f'Total de productos que tiene el precio a 0: {df_pz['description'].nunique()}')

Total de registros con "price" a 0: 70
Total de productos que tiene el precio a 0: 61


In [22]:
des_pz = df_pz['description'].unique()

In [23]:
df_dd_dz = df_dd.loc[df_dd['description'].apply(lambda x: x in des_pz, None)] # Se filtra en el DF sin duplicados por los productos que tienen precios a 0

In [24]:
df_dd_dz.loc[df_dd_dz['price'] > 0] # Se observa que esos productos pueden tener precio unitario

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
17,489436,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-01 09:06:00,1.45,13078,United Kingdom
71,489439,22065,CHRISTMAS PUDDING TRINKET POT,12,2009-12-01 09:28:00,1.45,12682,France
277,489465,48185,DOOR MAT FAIRY CAKE,4,2009-12-01 10:52:00,6.75,13767,United Kingdom
280,489465,85042,ANTIQUE LILY FAIRY LIGHTS,24,2009-12-01 10:52:00,4.25,13767,United Kingdom
317,C489503,21533,RETRO SPOT LARGE MILK JUG,-1,2009-12-01 11:04:00,4.95,16011,United Kingdom
...,...,...,...,...,...,...,...,...
1067168,581567,22624,IVORY KITCHEN SCALES,2,2011-12-09 11:56:00,8.50,16626,United Kingdom
1067170,581567,22625,RED KITCHEN SCALES,2,2011-12-09 11:56:00,8.50,16626,United Kingdom
1067214,581572,22624,IVORY KITCHEN SCALES,4,2011-12-09 12:08:00,8.50,16705,United Kingdom
1067284,581579,23480,MINI LIGHTS WOODLAND MUSHROOMS,8,2011-12-09 12:19:00,3.75,17581,United Kingdom


In [25]:
# Se guarda en un diccionario los productos con precio a 0 y cuántos registros hay
registros_0 = {prod:len(df_dd_dz.loc[(df_dd_dz['description'] == des_pz[i]) & (df_dd_dz['price'] == 0)]) for i,prod in enumerate(des_pz)}

In [26]:
[prod for prod,v in registros_0.items() if v == max(registros_0.values())] # Manual es el producto que más tiene el precio a 0

['Manual']

In [27]:
# En el DF sin duplicados, se filtra por facturas que NO han sido canceladas y por los productos 'Manual'
df_dd.loc[(~df_dd['invoice'].astype(str).str.startswith('C')) & (df_dd['description'] == 'Manual')]

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
11310,490300,M,Manual,1,2009-12-04 14:19:00,0.85,12970,United Kingdom
11311,490300,M,Manual,1,2009-12-04 14:19:00,0.21,12970,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.00,17231,United Kingdom
17386,490760,M,Manual,1,2009-12-08 09:49:00,10.00,14295,United Kingdom
17887,490881,M,Manual,1,2009-12-08 12:58:00,10.00,16210,United Kingdom
...,...,...,...,...,...,...,...,...
1051476,580645,M,Manual,1,2011-12-05 13:11:00,219.50,17857,United Kingdom
1051479,580646,M,Manual,800,2011-12-05 13:13:00,0.25,17857,United Kingdom
1056337,580884,M,Manual,1,2011-12-06 12:21:00,0.85,15907,United Kingdom
1056893,580956,M,Manual,4,2011-12-06 14:23:00,1.25,17841,United Kingdom


In [28]:
set_registros_0 = set(registros_0.values()) # Se convierte a set para eliminar duplicados
set_registros_0 # Los demás productos únicamente tienen 1 o 2 registros con precio a 0

{1, 2, 7}

***Concluciones***:
Se limpian los valores y se registran únicamente cantidades y precios positivos

In [9]:
df_dd = df_dd.loc[(df_dd['quantity'] > 0) & (df_dd['price'] > 0)]

### 5.5. **Filtrado por cancelaciones**
Las facturas que empiezan por 'C' son cancelaciones que han realizado los clientes, por lo que las excluiremos en la variable `Invoice`. En cambio, se construirá una nueva variable para no perder esa información ya que podría ser util.

In [10]:
df_dd['is_cancelled'] = df_dd['invoice'].astype(str).str.startswith('C')

In [11]:
# Se guarda en un DataFrame diferente aquellos clientes que NO han realizado ninguna cancelación
df_2 = df_dd[~df_dd['is_cancelled']].copy()

## 6. **Generación de la variable objetivo**
En el dataset no existe una variable para predecir si un cliente va a volver a comprar.

En este caso, la crearemos de la siguiente forma:
1. Dividiremos el dataset en dos grupos para definir un corte temporal:
    - Un set de `observación` donde se agruparán periodos de 21 meses 
    - Un set de `validación`donde se agrupará un periodo de los 3 últimos meses.
2. Construcción de la variable objetivo:
    - Si los clientes que han comprado en los últimos 3 meses coincide con los primeros 21, se guarda en otra variable (referenciando como 1 a los que repiten y 0 a los que no)


### 6.1. **Definición del corte temporal**

In [12]:
# Se define tiempo temporal
date_max = df_2['invoicedate'].max()
print(f'Fecha df_2 del dataset: {date_max}')

Fecha df_2 del dataset: 2011-12-09 12:50:00


In [13]:
# Se usa 3 meses como periodo de validacion
n_months = 3
date_thres = date_max - pd.DateOffset(months=n_months)
print(f'Fecha corte: {date_thres}')

Fecha corte: 2011-09-09 12:50:00


In [14]:
# Periodo de observacion: todo antes del corte
df_obs = df_2.loc[df_2['invoicedate'] < date_thres].copy()

# Periodo de validacion: últimos 3 meses (para construir el target)
df_val = df_2.loc[df_2['invoicedate'] >= date_thres].copy()

print(f"Registros en observación: {len(df_obs):,}")
print(f"Registros en validación: {len(df_val):,}")
print(f"Clientes únicos en observación: {df_obs['customer_id'].nunique():,}")
print(f"Clientes únicos en validación: {df_val['customer_id'].nunique():,}")

Registros en observación: 619,901
Registros en validación: 159,524
Clientes únicos en observación: 5,279
Clientes únicos en validación: 2,893


### 6.2. **Construcción del Target**

In [15]:
# Clientes que compraron en el perido de validación (son los que "repiten")
cus_re = df_val['customer_id'].unique()

# Lista de clientes del periodo de observación
cus_obs = df_obs['customer_id'].unique()

# Creación el target: 1 si el cliente volvío a comprar, 0 si no
target = pd.DataFrame({'customer_id': cus_obs})
target['target'] = target['customer_id'].isin(cus_re).astype(int)

print(f"\nDistribución de la variable objetivo:")
print(target['target'].value_counts())
print(f"\nTasa de recompra: {target['target'].mean():.2%}")


Distribución de la variable objetivo:
target
0    2985
1    2294
Name: count, dtype: int64

Tasa de recompra: 43.46%


## 7. **Construcción de Features RFM**
El objetivo de este apartado es construir una tabla de **features a nivel de cliente** que capture su comportamiento de compra durante el periodo de observación

### 7.1. **Variable base: Revenue**

In [16]:
df_dd['revenue'] = df_dd['quantity'] * df_dd['price']
df_obs['revenue'] = df_obs['quantity'] * df_obs['price']

### 7.2. **Agregación por cliente**
Se agrupa el dataset por `customer_id` y se calculan las siguientes features:
|Feature|Variable origen|Descripción|
|--|--|--|
|`recency`|`invoicedate`|Días desde la última compra hasta la fecha de corte|
|`frequency`|`invoice`|Número de facturas únicas (pedidos)|
|`monetary`|`revenue`|Gasto total en el periodo|
|`avg_basket`|`revenue`|Ticket medio por línea de producto|
|`n_products`|`stockcode`|Variedad de productos distintos comprados|
|`n_countries`|`country`|Número de países distintos|

In [61]:
rfm = df_obs.groupby('customer_id').agg(
    recency = ('invoicedate', lambda x: (date_thres - x.max()).days),
    frequency = ('invoice', 'nunique'),
    monetary = ('revenue', 'sum'),
    avg_basket = ('revenue', 'mean'),
    n_products = ('stockcode', 'nunique'),
    n_countries = ('country', 'nunique'),
    first_shop = ('invoicedate', 'min'), # Columna auxiliar
    last_shop = ('invoicedate', 'max') # Columna auxiliar
).reset_index()

In [62]:
# Creación de la antigüedad del cliente en días
rfm['customer_age_days'] = (date_thres - rfm['first_shop']).dt.days

In [63]:
# Intervalo medio entre compra
rfm['avg_days_between_orders'] = np.where(
    rfm['frequency'] > 1,
    rfm['customer_age_days'] / (rfm['frequency'] - 1),
    rfm['customer_age_days'] # Si solo compró una vez, se usa su antigUedad
)

In [64]:
# Se eliminan las columnas auxiliares
rfm = rfm.drop(columns=['first_shop', 'last_shop'])

## 8. **Dataset final: Features + Target**

In [65]:
# Dataset para el EDA (registros únicos por clientes con generación de nuevas Features)
df_rfm = rfm.merge(target, on='customer_id', how='inner')

print(f"\nShape del dataset para modelar: {df_rfm.shape}")
display(df_rfm.head(10))


Shape del dataset para modelar: (5279, 10)


,customer_id,recency,frequency,monetary,avg_basket,n_products,n_countries,customer_age_days,avg_days_between_orders,target
0,12346,234,12,77556.46,2281.072353,27,1,634,57.636364,0
1,12347,38,6,3402.39,20.746280,107,1,312,62.400000,1
2,12348,157,4,1709.40,35.612500,25,1,346,115.333333,1
3,12349,316,3,2671.14,26.187647,90,1,497,248.500000,1
4,12350,218,1,334.40,19.670588,17,1,218,218.000000,0
5,12351,283,1,300.93,14.330000,21,1,283,283.000000,0
6,12352,170,7,1905.61,34.028750,38,1,301,50.166667,1
7,12353,112,2,406.76,16.948333,23,1,317,317.000000,0
8,12354,140,1,1079.40,18.610345,58,1,140,140.000000,0
9,12355,122,2,947.61,27.074571,35,1,476,476.000000,0


In [66]:
# Se guarda com .csv
df_rfm.to_csv('./data/clean/online_retail_unique.csv', index=False)

In [67]:
target_concat = df_rfm[['customer_id','target']]
df_all = df_dd.merge(target_concat, how='left', on='customer_id')

In [68]:
# Dataset de modelado
df_all.to_csv('./data/clean/online_retail_clean_all.csv', index=False)